In [52]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import prince

from importlib.metadata import version
from pathlib import Path
from matplotlib.transforms import offset_copy

import io
import contextlib
from adjustText import adjust_text

In [53]:
%run LittRuP__import_functions.ipynb

In [54]:
# chemins vers fichiers Data et Images

BASE_DIR = Path.cwd()

if BASE_DIR.name == "Notebooks":
    BASE_DIR = BASE_DIR.parent

DAT_DIR = BASE_DIR / "Data"
IMG_DIR = BASE_DIR / "Images"
DOC_DIR = BASE_DIR / "Docs"

In [55]:
# import matrice profil thématique réduit des auteurs

matrix_reduced_profile = pd.read_csv(DAT_DIR / "LittRu_AC_matrix_reduced_profile.csv", sep=',', header=0)

In [56]:
# auteurs en index

matrix_reduced_profile = matrix_reduced_profile.set_index("Author")

**Pour construire les tableaux d’éléments interprétables sur le plan factoriel 1–2, quatre fichiers suffisent**

Auteurs :
- LittRu_AC_contributions_auteurs_plan_1-2.csv
- LittRu_AC_cos2_auteurs_plan_1-2.csv

Thèmes :
- LittRu_AC_contributions_themes_plan_1-2.csv
- LittRu_AC_cos2_themes_plan_1-2.csv

Les fichiers concernant les masses, les valeurs propres, les distances d_1^2, et tous les axes ne sont pas nécessaires pour cette sélection. 

Ils restent utiles pour vérifier ou reconstituer les calculs.

**Pour la construction des tableaux**

In [57]:
# format d'affichage des tableaux

format_tableau = {
    "Contribution axe 1": "{:.2f} %",
    "Contribution axe 2": "{:.2f} %",
    "Contribution plan 1-2": "{:.2f} %",
    "Cos2 axe 1": "{:.2%}",
    "Cos2 axe 2": "{:.2%}",
    "Cos2 plan 1-2": "{:.2%}"
}

**Procédure pour les auteurs**

In [58]:
# ============================================================
# 1. Import des contributions et des cos² des auteurs
# ============================================================

contrib_auteurs = pd.read_csv(
    DAT_DIR / "LittRu_AC_contributions_auteurs_plan_1-2.csv",
    index_col=0
)

cos2_auteurs = pd.read_csv(
    DAT_DIR / "LittRu_AC_cos2_auteurs_plan_1-2.csv",
    index_col=0
)

# Vérification de la structure des fichiers
print("Colonnes des contributions :")
print(contrib_auteurs.columns.tolist())

print("\nColonnes des cos2 :")
print(cos2_auteurs.columns.tolist())

Colonnes des contributions :
['Masse', 'Contribution axe 1 (%)', 'Contribution axe 2 (%)', 'Contribution plan 1-2 (%)']

Colonnes des cos2 :
['Distance au centre d2', 'Cos2 axe 1', 'Cos2 axe 2', 'Cos2 plan 1-2']


In [59]:
# ============================================================
# 2. Construction du tableau complet des auteurs
# ============================================================

auteurs_selection = pd.DataFrame(
    index=contrib_auteurs.index
)

auteurs_selection["Contribution axe 1"] = (
    contrib_auteurs["Contribution axe 1 (%)"]
)

auteurs_selection["Contribution axe 2"] = (
    contrib_auteurs["Contribution axe 2 (%)"]
)

auteurs_selection["Contribution plan 1-2"] = (
    contrib_auteurs["Contribution plan 1-2 (%)"]
)

auteurs_selection["Cos2 axe 1"] = (
    cos2_auteurs["Cos2 axe 1"]
)

auteurs_selection["Cos2 axe 2"] = (
    cos2_auteurs["Cos2 axe 2"]
)

auteurs_selection["Cos2 plan 1-2"] = (
    cos2_auteurs["Cos2 plan 1-2"]
)

# ============================================================
# 3. Seuils de sélection des auteurs
# ============================================================

n_auteurs = len(auteurs_selection)

seuil_contribution_auteurs = 100 / n_auteurs
seuil_cos2 = 0.25

print(
    "Nombre d'auteurs :",
    n_auteurs
)

print(
    "Contribution moyenne d'un auteur au plan 1-2 :",
    f"{seuil_contribution_auteurs:.2f} %"
)

# ============================================================
# 4. Sélection des auteurs interprétables
# ============================================================

condition_contribution_auteurs = (
    auteurs_selection["Contribution plan 1-2"]
    >= seuil_contribution_auteurs
)

condition_cos2_auteurs = (
    auteurs_selection["Cos2 plan 1-2"]
    >= seuil_cos2
)

auteurs_interpretables = auteurs_selection.loc[
    condition_contribution_auteurs
    & condition_cos2_auteurs
].copy()

auteurs_interpretables = auteurs_interpretables.sort_values(
    by=[
        "Cos2 plan 1-2",
        "Contribution plan 1-2"
    ],
    ascending=False
)

display(
    auteurs_interpretables.style.format(
        format_tableau
    )
)

print(
    "Nombre d'auteurs interprétables :",
    len(auteurs_interpretables)
)

Nombre d'auteurs : 40
Contribution moyenne d'un auteur au plan 1-2 : 2.50 %


,Contribution axe 1,Contribution axe 2,Contribution plan 1-2,Cos2 axe 1,Cos2 axe 2,Cos2 plan 1-2
Author,,,,,,
FADEÏEV,20.87 %,1.47 %,12.46 %,85.65%,4.64%,90.29%
POLEVOÏ,20.87 %,1.47 %,12.46 %,85.65%,4.64%,90.29%
FEDINE,13.34 %,0.97 %,7.98 %,80.86%,4.49%,85.34%
BERGHOLTS,17.00 %,0.38 %,9.79 %,80.92%,1.38%,82.30%
GUINZBOURG,0.09 %,12.65 %,5.54 %,0.63%,68.39%,69.02%
SOLJENITSYNE,1.56 %,31.71 %,14.64 %,4.07%,63.10%,67.17%
TOURGUENIEV,2.72 %,3.12 %,2.89 %,22.53%,19.83%,42.35%
GORKI,0.05 %,9.90 %,4.32 %,0.21%,34.09%,34.31%


Nombre d'auteurs interprétables : 8


In [43]:
len(auteurs_interpretables)

8

**Même procédure pour les thèmes**

In [26]:
# ============================================================
# 5. Import des contributions et des cos² des thèmes
# ============================================================

contrib_themes = pd.read_csv(
    DAT_DIR / "LittRu_AC_contributions_themes_plan_1-2.csv",
    index_col=0
)

cos2_themes = pd.read_csv(
    DAT_DIR / "LittRu_AC_cos2_themes_plan_1-2.csv",
    index_col=0
)

# Vérification de la structure des fichiers
print("Colonnes des contributions :")
print(contrib_themes.columns.tolist())

print("\nColonnes des cos2 :")
print(cos2_themes.columns.tolist())

Colonnes des contributions :
['Masse', 'Contribution axe 1 (%)', 'Contribution axe 2 (%)', 'Contribution plan 1-2 (%)']

Colonnes des cos2 :
['Distance au centre d2', 'Cos2 axe 1', 'Cos2 axe 2', 'Cos2 plan 1-2']


In [45]:
# ============================================================
# 6. Construction du tableau complet des thèmes
# ============================================================

themes_selection = pd.DataFrame(
    index=contrib_themes.index
)

themes_selection["Contribution axe 1"] = (
    contrib_themes["Contribution axe 1 (%)"]
)

themes_selection["Contribution axe 2"] = (
    contrib_themes["Contribution axe 2 (%)"]
)

themes_selection["Contribution plan 1-2"] = (
    contrib_themes["Contribution plan 1-2 (%)"]
)


themes_selection["Cos2 axe 1"] = (
    cos2_themes["Cos2 axe 1"]
)

themes_selection["Cos2 axe 2"] = (
    cos2_themes["Cos2 axe 2"]
)

themes_selection["Cos2 plan 1-2"] = (
    cos2_themes["Cos2 plan 1-2"]
)

# ============================================================
# 7. Seuils de sélection des thèmes
# ============================================================

n_themes = len(themes_selection)

seuil_contribution_themes = 100 / n_themes

print(
    "Nombre de thèmes :",
    n_themes
)

print(
    "Contribution moyenne d'un thème sur un axe :",
    f"{seuil_contribution_themes:.2f} %"
)

# ============================================================
# 8. Sélection des thèmes interprétables
# ============================================================

condition_contribution_themes = (
    themes_selection["Contribution plan 1-2"]
    >= seuil_contribution_themes
)

condition_cos2_themes = (
    themes_selection["Cos2 plan 1-2"]
    >= seuil_cos2
)

themes_interpretables = themes_selection.loc[
    condition_contribution_themes
    & condition_cos2_themes
].copy()

themes_interpretables = themes_interpretables.sort_values(
    by=[
        "Cos2 plan 1-2",
        "Contribution plan 1-2"
    ],
    ascending=False
)

display(
    themes_interpretables.style.format(
        format_tableau
    )
)

print(
    "Nombre de thèmes interprétables :",
    len(themes_interpretables)
)

Nombre de thèmes : 19
Contribution moyenne d'un thème sur un axe : 5.26 %


,Contribution axe 1,Contribution axe 2,Contribution plan 1-2,Cos2 axe 1,Cos2 axe 2,Cos2 plan 1-2
Theme,,,,,,
Guerre,37.20 %,2.32 %,22.07 %,80.98%,3.86%,84.84%
Patrie,45.30 %,2.25 %,26.63 %,77.19%,2.93%,80.12%
Société,2.53 %,32.33 %,15.46 %,5.43%,53.11%,58.53%
Souffrance,0.09 %,29.84 %,12.99 %,0.21%,53.36%,53.57%


Nombre de thèmes interprétables : 4


In [46]:
len(themes_interpretables)

4

**Vérifications**

In [47]:
print(
    "Sommes des contributions des auteurs :"
)

print(
    auteurs_selection[
        [
            "Contribution axe 1",
            "Contribution axe 2"
        ]
    ].sum()
)

# le résultat doit être proche de 100

Sommes des contributions des auteurs :
Contribution axe 1    100.0
Contribution axe 2    100.0
dtype: float64


In [48]:
print(
    contrib_auteurs[
        [
            "Contribution axe 1 (%)",
            "Contribution axe 2 (%)"
        ]
    ].sum()
)

print(
    contrib_themes[
        [
            "Contribution axe 1 (%)",
            "Contribution axe 2 (%)"
        ]
    ].sum()
)

Contribution axe 1 (%)    100.0
Contribution axe 2 (%)    100.0
dtype: float64
Contribution axe 1 (%)    100.0
Contribution axe 2 (%)    100.0
dtype: float64


In [49]:
print(
    auteurs_selection[
        [
            "Cos2 axe 1",
            "Cos2 axe 2",
            "Cos2 plan 1-2"
        ]
    ].min()
)

print(
    auteurs_selection[
        [
            "Cos2 axe 1",
            "Cos2 axe 2",
            "Cos2 plan 1-2"
        ]
    ].max()
)
# doit produire des valeurs comprises entre 0 et 1

Cos2 axe 1       5.249400e-09
Cos2 axe 2       8.022714e-06
Cos2 plan 1-2    1.109159e-02
dtype: float64
Cos2 axe 1       0.856514
Cos2 axe 2       0.683882
Cos2 plan 1-2    0.902870
dtype: float64


In [50]:
print(
    themes_selection[
        [
            "Cos2 axe 1",
            "Cos2 axe 2",
            "Cos2 plan 1-2"
        ]
    ].min()
)

print(
    themes_selection[
        [
            "Cos2 axe 1",
            "Cos2 axe 2",
            "Cos2 plan 1-2"
        ]
    ].max()
)

Cos2 axe 1       0.000593
Cos2 axe 2       0.000193
Cos2 plan 1-2    0.027887
dtype: float64
Cos2 axe 1       0.809817
Cos2 axe 2       0.533593
Cos2 plan 1-2    0.848423
dtype: float64


**Export des deux tableaux**

In [51]:
# ============================================================
# 9. Export des éléments interprétables
# ============================================================

auteurs_interpretables.to_csv(
    DAT_DIR / "LittRu_AC_auteurs_interpretables_plan_1-2.csv",
    encoding="utf-8-sig"
)

themes_interpretables.to_csv(
    DAT_DIR / "LittRu_AC_themes_interpretables_plan_1-2.csv",
    encoding="utf-8-sig"
)

print(
    "Fichiers CSV des éléments interprétables enregistrés."
)

Fichiers CSV des éléments interprétables enregistrés.
